In [ ]:
# MOSFET Parameters
Rds_on = 0.0014  # On-resistance in ohms
Vds = 60.0   # Drain-source voltage in volts
Idrain = 140.0    # On-current in amperes
ton_hs = 100e-9  # High-side turn-on time in seconds
toff_hs = 100e-9 # High-side turn-off time in seconds
ton_ls = 0e-9  # Low-side turn-on time in seconds - this value is currently not used since we assume zero voltage switching during turn-on
toff_ls = 0e-9 # Low-side turn-off time in seconds - this value is currently not used since we assume zero voltage switching during turn-off
f_sw = 5e3    # Switching frequency in hertz


qoss = 262e-9   # Output charge in coulombs
qg_tot = 250e-9   # Input charge in coulombs when driven to 15 V
vgate = 15.0   # Gate drive voltage in volts
duty_cycle = 0.9  # Duty cycle (0 to 1)
dead_time = 200e-9  # Dead time in seconds

# The half bridge has three modes of operation: PWMing, pulled low, and tri-state.

In [5]:
''' Low side MOSFET Loss Calculations '''

# The low side MOSFET is on for the 100% of the pulled low and duty% of the PWMing modes. 
P_conduction_ls = Rds_on * Idrain**2 * (1 + duty_cycle) / 3

# There are two components of switching loss: charging and discharging the output capacitance, 
# and the transition losses during turn-on and turn-off. Since a switching event happens with 
# dead time in between, and the switch node is pulled low during dead time, we can assume zero
# voltage switching and only consider the capacitance losses
P_output_switching_ls = 2 * qoss * Vds * f_sw
P_input_switching_ls = 2 * qg_tot * vgate * f_sw

# Deadtime loss
P_deadtime_ls = 0.7 * Idrain * dead_time * f_sw * 2

# If switching losses are significant, then we should also consider output losses during the tristated phase.
# Are there also conduction losses during tristating?

P_total_ls = P_conduction_ls + P_output_switching_ls + P_input_switching_ls + P_deadtime_ls

print("Low Side MOSFET Losses:")
print(f"Conduction Loss: {P_conduction_ls:.2f} W")
print(f"Output Capacitance Loss: {P_output_switching_ls:.2f} W")
print(f"Input Capacitance Loss: {P_input_switching_ls:.2f} W")
print(f"Deadtime Loss: {P_deadtime_ls:.2f} W")

Low Side MOSFET Losses:
Conduction Loss: 17.38 W
Output Capacitance Loss: 0.16 W
Input Capacitance Loss: 0.04 W
Deadtime Loss: 0.20 W


In [6]:
''' High side MOSFET Loss Calculations '''

# The high side MOSFET is on for duty% of the PWMing mode only.
P_conduction_hs = Rds_on * Idrain**2 * duty_cycle / 3

# There are two components of switching loss here: charging and discharging the output capacitance, 
# and the transition losses during turn-on and turn-off. Dead time losses do not apply here since
# the current is traveling through the low side MOSFET during dead time.
P_output_switching_hs = 2 * qoss * Vds * f_sw
P_input_switching_hs = 2 * qg_tot * vgate * f_sw
P_transition_hs = (Vds * Idrain / 2) * (ton_hs + toff_hs) * f_sw

P_total_hs = P_conduction_hs + P_output_switching_hs + P_input_switching_hs + P_transition_hs

print("\nHigh Side MOSFET Losses:")
print(f"Conduction Loss: {P_conduction_hs:.2f} W")
print(f"Output Capacitance Loss: {P_output_switching_hs:.2f} W")
print(f"Input Capacitance Loss: {P_input_switching_hs:.2f} W")
print(f"Transition Loss: {P_transition_hs:.2f} W")


High Side MOSFET Losses:
Conduction Loss: 8.23 W
Output Capacitance Loss: 0.16 W
Input Capacitance Loss: 0.04 W
Transition Loss: 4.20 W
